In [ ]:
import pandas as pd
import json 
import pickle
import pandas as pd

from jestr.utils.eval import borda_count, get_target, convert_rank_to_hit_rates

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# 1 Input data

## 1.1 JESTR input data 
1. spectra_metadata_test.tsv
2. identifier_to_candidates_test.json, if not specified will be retrieve from huggingace
For running JESTR on complete MassSpecGym run unlabel.ipynb, will create spectra_metadata.tsv

In [ ]:
spectra = pd.read_csv("tests/data/spectra_metadata_test.tsv", sep="\t")
display(spectra.head(1))

## 1.2 JESTR with FAISS input data: 
1. unlabeled_spectra_test.json: list of spectra with metadata
2. list_of_lists_of_candidates_test.json: a list of candidate sets for each spectrum

Both input lists have a certain order so the first spectrum correspons to the first candidate list etc.

Example input files are provided in tests/data/ to run JESTR with FAISS on the full MassSpecGym dataset run unlabel-ipynb,
this will store the complete files in the data folder.


In [ ]:
with open("tests/data/unlabeled_spectra_test.json", "r") as f:
    spectra = json.load(f)
display(spectra[0])

with open("tests/data/list_of_lists_of_candidates_test.json", "r") as f:
    candidates = json.load(f)
    display(candidates[0])

# 2 Run program

## 2.1 Running JESTR
```bash
cd jestr
python -m inference
```

## 2.2 Running JESTR with FAISS
```bash
cd jestr
python -m faiss_precompute
python -m faiss_inference
```

In [ ]:
def display_hit_rates(dataframe):
    dataframe["rank"] = dataframe.apply(
        lambda row: borda_count(row["candidates"], [row["scores"]], get_target(row["candidates"], row["labels"])),
        axis=1,
    )

    hit_rate_cols = dataframe.apply(
        lambda row: convert_rank_to_hit_rates(row, "rank", top_k=[1, 5, 20]),
        axis=1,
    )
    dataframe = pd.concat([dataframe, hit_rate_cols], axis=1)

    hit_rate_summary = dataframe[["rank-hit_rate@1", "rank-hit_rate@5", "rank-hit_rate@20"]].mean().to_dict()
    ranks = dataframe["rank"].tolist()
    mean_hit_rates = dataframe[["rank-hit_rate@1", "rank-hit_rate@5", "rank-hit_rate@20"]].mean()
    return mean_hit_rates

# 3 Output

## 3.1 FAISS Pipeline:

In [ ]:
with open("experiments/20260321_FAISS_sample_run_1/result_unlabeled_spectra_test.pkl", "rb") as f:
    result_faiss = pickle.load(f)
result_faiss_df = pd.DataFrame(result_faiss)
result_faiss_df.head(1)

In [ ]:
display_hit_rates(result_faiss_df)

## 3.2 Regular Pipeline:

In [ ]:
with open("test_results/20260713_JESTR_sample_run/result_spectra_metadata_test.pkl", "rb") as f:   
    result_faiss = pickle.load(f)
result_faiss_df = pd.DataFrame(result_faiss)
result_faiss_df.head(1)

In [ ]:
display_hit_rates(result_faiss_df)